# Kidney Stone 2D Segmentation — KSSD2025
Bu notebook Google Colab'da projectni 0 dan Gradio'gacha run qilish uchun tayyorlangan.


## 1. Google Drive'ni ulash


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Project papkasiga kirish
Agar papka nomi yoki joyi boshqacha bo'lsa `PROJECT_DIR` ni o'zgartiring.


In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/kidney_stone_project'
%cd $PROJECT_DIR
!pwd
!ls


## 3. Kutubxonalarni o'rnatish


In [ ]:
!pip install -q -r requirements.txt


## 4. GPU/CUDA tekshirish


In [ ]:
!nvidia-smi
!python src/check_environment.py


## 5. KSSD2025 datasetini Kaggle'dan yuklash
Agar Kaggle login talab qilsa, pastdagi login cellini ishlating.


In [ ]:
# Faqat login so'ralsa uncomment qiling:
# import kagglehub
# kagglehub.login()


In [ ]:
!python src/download_data.py --config configs/config.yaml


## 6. Real dataset strukturasini ko'rish


In [ ]:
!python src/inspect_data.py --config configs/config.yaml --limit 80


## 7. Image-mask pairing va split yaratish


In [ ]:
!python src/prepare_data.py --config configs/config.yaml


## 8. Bitta sample CT + maskni ko'rish


In [ ]:
!python src/visualize_sample.py --config configs/config.yaml
from IPython.display import Image, display
display(Image(filename='results/figures/sample_overlay.png'))


## 9. 2D U-Net training
Default: 12 epoch, batch=8, 256×256, AMP, early stopping.


In [ ]:
!python src/train.py --config configs/config.yaml


### Training uzilib qolsa davom ettirish


In [ ]:
# Kerak bo'lsa run qiling:
# !python src/train.py --config configs/config.yaml --resume


## 10. Test evaluation


In [ ]:
!python src/evaluate.py --config configs/config.yaml
!cat results/metrics/test_summary.json


## 11. Bitta image uchun prediction
`IMAGE_PATH` ni real test image pathiga almashtiring.


In [ ]:
import pandas as pd
test_df = pd.read_csv('data/splits/test.csv')
IMAGE_PATH = test_df.iloc[0]['image']
print(IMAGE_PATH)
!python src/predict.py "$IMAGE_PATH" --config configs/config.yaml


In [ ]:
from IPython.display import Image, display
from pathlib import Path
overlays = sorted(Path('results/predictions').glob('*_overlay.png'), key=lambda p: p.stat().st_mtime)
if overlays:
    display(Image(filename=str(overlays[-1])))


## 12. Gradio app


In [ ]:
!python app/app.py
